# Main analysis

In [3]:
MESSAGES_WITH_MISTRAL_PREDICTIONS = Path('insert_path_to_messages_with_mistral_predictions')

Read and prepare data

In [ ]:
df = pd.read_csv(MESSAGES_WITH_MISTRAL_PREDICTIONS)
print(f'Corpus size: {len(df)}')

Corpus size: 61346


In [3]:
# Clean predicted labels

df['clean_labels'] = df['clean_labels'].apply(ast.literal_eval)
df_exploded = df.explode('clean_labels')

# --- Merge labels in incorrect format ---
label_corrections = {
    'Islamophobia': 'Antisemitism and islamophobia',
    'Islamophobia and antisemitism': 'Antisemitism and islamophobia',
    'Nationalism and white Supremacy': 'Nationalism and white supremacy'
    }
df_exploded[df_exploded['mistral_agent_predictions'] == 'Nationalism and white Supremacy'] = 'Nationalism and white supremacy'
df_exploded['clean_labels'] = df_exploded['clean_labels'].replace(label_corrections)

# --- Reset index ---
df_exploded = df_exploded[df_exploded['clean_labels'].notna()]
df_exploded = df_exploded.drop_duplicates()
df_exploded = df_exploded.reset_index(drop=True)

## Results

### Corpus description

In [4]:
# Topic representation in the corpus
# --- Print label distribution table ---
df_exploded['clean_labels'].value_counts(normalize=True,dropna=False)

clean_labels
None of the above                              0.348164
Online coordination                            0.132548
Anti-establishment, lies and disinformation    0.088766
UK politics and Brexit                         0.080752
Nationalism and white supremacy                0.072321
Proud Boys/Antifa and the manosphere           0.069650
Antisemitism and islamophobia                  0.057935
Offline activism, mobilization and violence    0.048084
US politics and Trumpism                       0.038918
Anti-left                                      0.036957
Immigration                                    0.025904
Name: proportion, dtype: float64

In [5]:
# Exclude 'None of the above'

df_filtered = df_exploded[df_exploded['clean_labels'] != 'None of the above'].copy()

# Display the value counts of the filtered data
display(df_filtered['clean_labels'].value_counts(normalize=True))

clean_labels
Online coordination                            0.203346
Anti-establishment, lies and disinformation    0.136178
UK politics and Brexit                         0.123884
Nationalism and white supremacy                0.110950
Proud Boys/Antifa and the manosphere           0.106852
Antisemitism and islamophobia                  0.088881
Offline activism, mobilization and violence    0.073766
US politics and Trumpism                       0.059705
Anti-left                                      0.056697
Immigration                                    0.039741
Name: proportion, dtype: float64

In [6]:
# Top-5 most co-occuring topic pairs
# Calculate the co-occurrence matrix of topics within messages
# Group by message and get the list of labels for each message
message_labels = df_filtered.groupby('message')['clean_labels'].apply(list)

# Create a list of all unique labels
all_labels = df_filtered['clean_labels'].unique()

# Initialize an empty co-occurrence matrix
co_occurrence_matrix = pd.DataFrame(0, index=all_labels, columns=all_labels, dtype=int)

# Iterate through each message's labels and update the co-occurrence matrix
for labels in message_labels:
    unique_labels = set(labels)  # Avoid double-counting within the same message
    for label1 in unique_labels:
        for label2 in unique_labels:
            if label1 != label2:  # Skip self-co-occurrence
                co_occurrence_matrix.loc[label1, label2] += 1

# Get the top 5 most co-occurring topic pairs
# Unstack the matrix to get a Series of pairs and their counts
co_occurrence_series = co_occurrence_matrix.unstack()

# Convert to a DataFrame for easier filtering
co_occurrence_df = co_occurrence_series.reset_index()
co_occurrence_df.columns = ['Topic 1', 'Topic 2', 'Count']

# Exclude self-co-occurrences
co_occurrence_df = co_occurrence_df[co_occurrence_df['Topic 1'] != co_occurrence_df['Topic 2']]

# Ensure each unordered pair appears only once (e.g., treat (A, B) same as (B, A))
co_occurrence_df['Pair'] = co_occurrence_df.apply(lambda x: tuple(sorted([x['Topic 1'], x['Topic 2']])), axis=1)
co_occurrence_df = co_occurrence_df.groupby('Pair', as_index=False)['Count'].sum()

# Sort and get the top 5 pairs
top_5_pairs = co_occurrence_df.sort_values(by='Count', ascending=False).head(5)

print("Top 5 Most Co-occurring Topic Pairs (unique):")
display(top_5_pairs)


Top 5 Most Co-occurring Topic Pairs (unique):


,Pair,Count
40,"(Online coordination, UK politics and Brexit)",2714
5,"(Anti-establishment, lies and disinformation, ...",2674
7,"(Anti-establishment, lies and disinformation, ...",2134
41,"(Online coordination, US politics and Trumpism)",2062
35,"(Offline activism, mobilization and violence, ...",1866


### Topical landscape

In [7]:
# Calculate the total count for each topic across all channels (in the filtered data)
total_topic_counts = df_filtered['clean_labels'].value_counts()

# Calculate the count of each topic within each channel
username_topic_counts = df_filtered.groupby(['username', 'clean_labels']).size().unstack(fill_value=0)

# Calculate the percentage of each topic's total count that comes from each channel
# Divide the username_topic_counts by the total_topic_counts, aligning by topic (columns)
topic_percentage_per_channel = username_topic_counts.divide(total_topic_counts, axis=1).fillna(0)

# To find topics concentrated in a few channels, we can look at the maximum percentage any single channel contributes to a topic.
# A high maximum percentage suggests the topic is heavily concentrated in that channel.
topic_concentration_by_max_channel = topic_percentage_per_channel.max().sort_values(ascending=False)

print("Topics sorted by maximum percentage contributed by a single channel (higher percentage = more concentrated):")
display(topic_concentration_by_max_channel)

# Alternatively, we can look at the standard deviation of the percentages across channels.
# A higher standard deviation suggests the topic distribution is more spread out across channels, while a lower standard deviation (closer to 0) suggests it's more evenly distributed.
# This is the opposite of the previous approach, so we'll sort in ascending order for "more evenly spread".
topic_evenness_by_std = topic_percentage_per_channel.std().sort_values(ascending=False)

print("\nTopics sorted by evenness across channels (lower standard deviation = more evenly spread):")
display(topic_evenness_by_std)

Topics sorted by maximum percentage contributed by a single channel (higher percentage = more concentrated):


clean_labels
UK politics and Brexit                         0.599848
US politics and Trumpism                       0.409005
Immigration                                    0.327815
Offline activism, mobilization and violence    0.323394
Anti-establishment, lies and disinformation    0.320817
Anti-left                                      0.308355
Online coordination                            0.293612
Proud Boys/Antifa and the manosphere           0.203730
Antisemitism and islamophobia                  0.171743
Nationalism and white supremacy                0.096916
dtype: float64


Topics sorted by evenness across channels (lower standard deviation = more evenly spread):


clean_labels
UK politics and Brexit                         0.080477
US politics and Trumpism                       0.055659
Online coordination                            0.047513
Offline activism, mobilization and violence    0.046781
Immigration                                    0.046280
Anti-establishment, lies and disinformation    0.045151
Anti-left                                      0.044186
Proud Boys/Antifa and the manosphere           0.036317
Antisemitism and islamophobia                  0.028401
Nationalism and white supremacy                0.021745
dtype: float64

### Channel-level descriptions

In [8]:
# Compute each channel's topic distribution vector
# Normalize the counts to get proportions
channel_topic_distribution_normalized = username_topic_counts.divide(username_topic_counts.sum(axis=1), axis=0)

print("Channel Topic Distribution Vectors (Normalized):")
display(channel_topic_distribution_normalized)

Channel Topic Distribution Vectors (Normalized):


clean_labels,"Anti-establishment, lies and disinformation",Anti-left,Antisemitism and islamophobia,Immigration,Nationalism and white supremacy,"Offline activism, mobilization and violence",Online coordination,Proud Boys/Antifa and the manosphere,UK politics and Brexit,US politics and Trumpism
username,,,,,,,,,,
AltKnight,0.154348,0.071739,0.067391,0.056522,0.128261,0.082609,0.173913,0.176087,0.021739,0.067391
BasedAmyNews,0.000000,0.125000,0.000000,0.000000,0.000000,0.125000,0.625000,0.000000,0.000000,0.125000
BlackonWhiteCrime,0.273810,0.011905,0.071429,0.119048,0.261905,0.154762,0.047619,0.035714,0.011905,0.011905
BloodAndSoil,0.000000,0.000000,0.000000,0.000000,0.818182,0.000000,0.181818,0.000000,0.000000,0.000000
Brexiteers,0.151575,0.059226,0.045545,0.043384,0.019982,0.079928,0.223762,0.068047,0.271287,0.037264
...,...,...,...,...,...,...,...,...,...,...
untitledredpills,0.111111,0.013072,0.130719,0.013072,0.274510,0.078431,0.287582,0.071895,0.006536,0.013072
wawaryanwoman,0.035294,0.011765,0.047059,0.011765,0.752941,0.000000,0.070588,0.058824,0.000000,0.011765
weebposting,0.049180,0.049180,0.180328,0.000000,0.196721,0.065574,0.163934,0.295082,0.000000,0.000000


In [9]:
from scipy.stats import entropy

# Calculate Shannon entropy for each row (channel)
# We need to use the normalized distribution (proportions)
shannon_entropy = entropy(channel_topic_distribution_normalized, axis=1, base=2)

# Create a Series with channel usernames as index
shannon_entropy_series = pd.Series(shannon_entropy, index=channel_topic_distribution_normalized.index)

# Sort the results for better readability
shannon_entropy_series = shannon_entropy_series.sort_values(ascending=False)

print("Shannon Entropy of Topic Distribution per Channel:")
display(shannon_entropy_series)

Shannon Entropy of Topic Distribution per Channel:


username
ResistMarxism         3.149259
AltKnight             3.124909
WhiteIsRight          3.087110
PoliticalMemes        3.059563
proudboysbritannia    2.998977
                        ...   
HitlerismWorldwide    0.468996
Hitler_was_Right      0.330277
cristoariano          0.000000
backstagenews         0.000000
pishroofficial        0.000000
Length: 63, dtype: float64

In [10]:
# Calculate the average topic distribution for the low-entropy users
lowest_entropy_users = shannon_entropy_series.tail(10).index
average_low_entropy_distribution = channel_topic_distribution_normalized.loc[lowest_entropy_users].mean()

print("Average Topic Distribution for Channels with Lowest Shannon Entropy:")
display(average_low_entropy_distribution.sort_values(ascending=False))

Average Topic Distribution for Channels with Lowest Shannon Entropy:


clean_labels
Nationalism and white supremacy                0.514813
Online coordination                            0.328862
Proud Boys/Antifa and the manosphere           0.093435
Anti-left                                      0.050000
Antisemitism and islamophobia                  0.012890
Anti-establishment, lies and disinformation    0.000000
Immigration                                    0.000000
Offline activism, mobilization and violence    0.000000
UK politics and Brexit                         0.000000
US politics and Trumpism                       0.000000
dtype: float64

## Network analyses

The edge list is extracted from the dataset above by selecting forwarded messages and merging with channel metadata from the Pushshift dataset. Topics prevalence vectors are already included in as node attributes in ```graph.gexf```, and they were defined with the code belwo (which overwrites previous versions of ```graph.gexf```).

In [11]:
G = nx.read_gexf('graph.gexf')

In [12]:
# Evaluate node attributes as literal

for _, attributes in G.nodes(data=True):
    for key, value in attributes.items():
        if isinstance(value, str):
            try:
                attributes[key] = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                pass

### Topic prevalence

In [13]:
# Calculate the count of each topic within each channel
username_topic_counts = df_filtered.groupby(['username', 'clean_labels']).size().unstack(fill_value=0)

topic_prevalence = username_topic_counts.div(username_topic_counts.sum(axis=1), axis=0)

In [14]:
# Convert the topic prevalence DataFrame to a dictionary of dictionaries for easier access
channel_topic_dict = topic_prevalence.T.to_dict()

# Add topic prevalence as node attributes to the graph G
for node in G.nodes():
    if node in channel_topic_dict:
        G.nodes[node]['topic_distribution'] = channel_topic_dict[node]
    else:
        # Handle cases where a node in the graph might not have topic data (optional)
        # print(f"Warning: No topic data found for node: {node}")
        G.nodes[node]['topic_distribution'] = None # Or some default value

print("Topic prevalence vectors added as node attributes.")

# Verify by checking attributes of a few nodes (optional)
for i, node in enumerate(G.nodes()):
    if i < 5:
        print(f"Node {node}: {G.nodes[node].get('topic_distribution')}")
    else:
        break

Topic prevalence vectors added as node attributes.
Node AltKnight: {'Anti-establishment, lies and disinformation': 0.15434782608695652, 'Anti-left': 0.07173913043478261, 'Antisemitism and islamophobia': 0.06739130434782609, 'Immigration': 0.05652173913043478, 'Nationalism and white supremacy': 0.1282608695652174, 'Offline activism, mobilization and violence': 0.08260869565217391, 'Online coordination': 0.17391304347826086, 'Proud Boys/Antifa and the manosphere': 0.17608695652173914, 'UK politics and Brexit': 0.021739130434782608, 'US politics and Trumpism': 0.06739130434782609}
Node Brexiteers: {'Anti-establishment, lies and disinformation': 0.15157515751575157, 'Anti-left': 0.05922592259225923, 'Antisemitism and islamophobia': 0.04554455445544554, 'Immigration': 0.043384338433843386, 'Nationalism and white supremacy': 0.01998199819981998, 'Offline activism, mobilization and violence': 0.07992799279927992, 'Online coordination': 0.22376237623762377, 'Proud Boys/Antifa and the manospher

### Shannon entropy

In [15]:
from scipy.stats import entropy

# Calculate Shannon entropy for each row (channel)
# We need to use the normalized distribution (proportions)
shannon_entropy = entropy(topic_prevalence, axis=1, base=2)

# Create a Series with channel usernames as index
shannon_entropy_series = pd.Series(shannon_entropy, index=topic_prevalence.index)

# Sort the results for better readability
shannon_entropy_series = shannon_entropy_series.sort_values(ascending=False)

print("Shannon Entropy of Topic Distribution per Channel:")
display(shannon_entropy_series)

Shannon Entropy of Topic Distribution per Channel:


username
ResistMarxism         3.149259
AltKnight             3.124909
WhiteIsRight          3.087110
PoliticalMemes        3.059563
proudboysbritannia    2.998977
                        ...   
HitlerismWorldwide    0.468996
Hitler_was_Right      0.330277
cristoariano          0.000000
backstagenews         0.000000
pishroofficial        0.000000
Length: 63, dtype: float64

### Topic assortativity

In [16]:
# Get the list of topics
topics = topic_prevalence.columns

# Dictionary to store assortativity coefficients for each topic
topic_assortativity = {}

# Compute assortativity for each topic
for topic in topics:
    # Create a dictionary of node attributes for the current topic
    node_attributes = {node: data['topic_distribution'].get(topic, 0) for node, data in G.nodes(data=True) if data.get('topic_distribution') is not None}

    # Ensure that there are nodes with non-zero attribute values for the topic
    if any(value > 0 for value in node_attributes.values()):
        # Compute numeric assortativity coefficient
        # Note: nx.numeric_assortativity_coefficient requires all nodes in the graph to have the attribute
        # We can filter the graph or pad with zeros/NaNs, but for simplicity and to focus on
        # nodes with topic data, we can create a subgraph or use a similar approach if needed.
        # Let's use the attribute name directly if all relevant nodes have the attribute.
        # Assuming 'topic_distribution' is a dictionary and we are accessing 'topic' within it.

        # Create a mapping of node to the specific topic prevalence
        topic_attribute_map = {node: data['topic_distribution'].get(topic, 0.0) for node, data in G.nodes(data=True) if data.get('topic_distribution') is not None}

        # Calculate assortativity using the attribute map
        # Need to ensure the nodes in the map are in the graph G
        nodes_in_graph = set(G.nodes())
        filtered_topic_attribute_map = {node: value for node, value in topic_attribute_map.items() if node in nodes_in_graph}

        if filtered_topic_attribute_map:
             # Create a temporary graph with the single attribute for assortativity calculation
            G_temp = G.copy()
            for node, value in filtered_topic_attribute_map.items():
                G_temp.nodes[node][topic] = value

            try:
                # Use the attribute name for calculation
                assortativity_coeff = nx.numeric_assortativity_coefficient(G_temp, topic)
                topic_assortativity[topic] = assortativity_coeff
            except Exception as e:
                print(f"Could not compute assortativity for topic '{topic}': {e}")
                topic_assortativity[topic] = None # Or np.nan

    else:
        print(f"No nodes with prevalence > 0 for topic '{topic}'. Cannot compute assortativity.")
        topic_assortativity[topic] = None # Or np.nan


# Print the assortativity coefficients
print("\nTopic Assortativity Coefficients:")
for topic, coeff in topic_assortativity.items():
    print(f"{topic}: {coeff:.4f}" if coeff is not None else f"{topic}: N/A")


Topic Assortativity Coefficients:
Anti-establishment, lies and disinformation: 0.3972
Anti-left: 0.5760
Antisemitism and islamophobia: 0.5314
Immigration: 0.2909
Nationalism and white supremacy: 0.7053
Offline activism, mobilization and violence: 0.4802
Online coordination: 0.5212
Proud Boys/Antifa and the manosphere: 0.6180
UK politics and Brexit: 0.5045
US politics and Trumpism: 0.4953


In [17]:
# (Over)Write objects

topic_prevalence.to_csv("topic_prevalence.csv", index=True)
nx.write_gexf(G, 'graph.gexf')
df_filtered.to_csv('df_filtered.csv', index=False)